# Eigenmode simulation of coupled qubits

## Rendering the design

## Two qubit example with resonators and a coupler


In [ ]:
%load_ext autoreload
%autoreload 2
from copy import deepcopy
import names as n
import design as d
from qdesignoptimizer.utils.chip_generation import create_chip_base

import mini_studies as ms
import optimization_targets as ot

import parameter_targets as pt
import plot_settings as ps

from qdesignoptimizer.design_analysis import (
    DesignAnalysis,
    DesignAnalysisState,
    fetch_params_and_design_variables,
)
from qdesignoptimizer.utils.utils import get_save_path
from qdesignoptimizer.utils.names_parameters import param, param_nonlin

from qdesignoptimizer.utils.utils import close_ansys

close_ansys()

design, gui = create_chip_base(n.CHIP_NAME, d.chip_type, open_gui=True)
d.render_qiskit_metal_design(design, gui)

## Running the study in partitions 


In [ ]:
# Iteratively optimize the two qubit chip with the following divisions:


from matplotlib import pyplot as plt


RES_QB_CPLR = "res_qb_cplr"
QB_CPLR_QB = "qb_cplr_qb"
CPLR_QB_RES = "cplr_qb_res"

nbr_iterations = 11
group_passes = 17
delta_f = 0.001

run_counter = 0
optimization_results = []
minimization_results = []

PLOT_SETTINGS = ps.PLOT_SETTINGS_TWO_QB
RENDER_QISKIT_METAL = lambda design: d.render_qiskit_metal_design(design, gui)
design_analysis_state = DesignAnalysisState(
    design, RENDER_QISKIT_METAL, pt.PARAM_TARGETS
)
groups = [n.NBR_1, n.NBR_2]
qubit_freq_1_for_1_in = []
qubit_freq_1_for_1_out = []
qubit_freq_1_for_2_in = []
qubit_freq_1_for_2_out = []
qubit_freq_1_from_opt_results_for_1out = []
qubit_freq_1_from_opt_results_for_2out = []

state_partition = DesignAnalysisState(design, RENDER_QISKIT_METAL, pt.PARAM_TARGETS)

for i in range(nbr_iterations):
    # Use a clean state from previous iteration to demonstrate that all subsystems can be optimized in parallel
    prev_iteration_variables = deepcopy(design_analysis_state.design.variables)
    prev_iteration_system_optimized_params = deepcopy(
        design_analysis_state.system_optimized_params
    )
    new_iteration = True
    for section in [RES_QB_CPLR, CPLR_QB_RES, QB_CPLR_QB]:

        MINI_STUDY = ms.get_mini_study_2qb_resonator_coupler_partitioned(section)

        if section == RES_QB_CPLR:
            exclude_param_from_update = [
                param(n.COUPLER_12, n.FREQ),
                param_nonlin(n.COUPLER_12, n.COUPLER_12),
                param_nonlin(n.COUPLER_12, n.QUBIT_1),
            ]
            include_param_in_update = [
                param(n.QUBIT_1, n.FREQ),
                param(n.RESONATOR_1, n.FREQ),
                param_nonlin(n.QUBIT_1, n.QUBIT_1),
                param_nonlin(n.QUBIT_1, n.RESONATOR_1),
            ]
        elif section == CPLR_QB_RES:
            exclude_param_from_update = [
                param(n.COUPLER_12, n.FREQ),
                param_nonlin(n.COUPLER_12, n.COUPLER_12),
                param_nonlin(n.COUPLER_12, n.QUBIT_2),
            ]
            include_param_in_update = [
                param(n.QUBIT_2, n.FREQ),
                param(n.RESONATOR_2, n.FREQ),
                param_nonlin(n.QUBIT_2, n.QUBIT_2),
                param_nonlin(n.QUBIT_2, n.RESONATOR_2),
            ]
        elif section == QB_CPLR_QB:
            exclude_param_from_update = [
                param(n.QUBIT_1, n.FREQ),
                param_nonlin(n.QUBIT_1, n.QUBIT_1),
                param(n.QUBIT_2, n.FREQ),
                param_nonlin(n.QUBIT_2, n.QUBIT_2),
            ]
            include_param_in_update = [
                param(n.COUPLER_12, n.FREQ),
                param_nonlin(n.COUPLER_12, n.COUPLER_12),
                param_nonlin(n.COUPLER_12, n.QUBIT_1),
                param_nonlin(n.COUPLER_12, n.QUBIT_2),
            ]
        state_partition.exclude_param_from_update = exclude_param_from_update

        # Include all targets which will be used in all partitions such that
        # the update of the design variables know which parameters will change
        opt_targets = ot.get_opt_targets_2qubits_resonator_coupler(
            groups=groups,
            opt_target_qubit_freq=True,
            opt_target_qubit_anharm=True,
            opt_target_resonator_freq=True,
            opt_target_resonator_qubit_chi=True,
            opt_target_coupler_freq=True,
            opt_target_coupler_anharm=True,
            opt_target_coupler_qubit_chi=True,
        )

        design_analysis = DesignAnalysis(
            state_partition,
            mini_study=MINI_STUDY,
            opt_targets=opt_targets,
            save_path=get_save_path("out/", n.CHIP_NAME),
            update_design_variables=False,
            plot_settings=PLOT_SETTINGS,
            partitioned_optimization=True,
        )
        # Continue from optimization and minimization results from previous partitions
        design_analysis.optimization_results = optimization_results
        design_analysis.minimization_results = minimization_results

        design_analysis.update_nbr_passes(group_passes)
        design_analysis.update_delta_f(delta_f)

        # Use a clean state from previous iteration to demonstrate that all subsystems can be optimized in parallel
        design_analysis.optimize_target(
            deepcopy(prev_iteration_variables),
            deepcopy(prev_iteration_system_optimized_params),
        )
        design_analysis.screenshot(gui=gui, run=run_counter)

        # Store optimization and minimization results for the next iteration
        optimization_results = design_analysis.optimization_results
        minimization_results = design_analysis.minimization_results
        state_partition.system_optimized_params = (
            design_analysis.system_optimized_params
        )

        # Please note that the last iteration of saved .npy files will not be formatted correctly since
        # they are saved before the merging of the iteration results. We currently recomment running 1 extra iteration.
        # Please also note that the dynamic plotting while the optimiztion is running plots partly outdated information.
        # We should disable the plotting in the design_analysis and just plot the optimization_results
        fetch_params_and_design_variables(
            design_analysis_state,
            state_partition,
            opt_targets,
            include_param_in_update,
            optimization_results,
            new_iteration,
        )
        new_iteration = False

        run_counter += 1

## Results

### Eigenmodes

In [ ]:
design_analysis.get_eigenmode_results()

### Cross Kerr

In [ ]:
design_analysis.get_cross_kerr_matrix(iteration=-1)

## Update parameters

In [ ]:
design_analysis.overwrite_parameters()

## Close